# NVFP4 + FP8 attention + text-only, and what it costs a drafter

`nvidia/Gemma-4-31B-IT-NVFP4` is more conservative than its name suggests. NVFP4 is a
4-bit weight *and* activation format, but the build excludes 63 modules — and 60 of them
are the attention blocks, listed one layer at a time. **Only the MLPs were quantized**,
which leaves unquantized attention as the largest single block in the file.

That is a defensible choice: NVFP4 quantizes activations too, and attention activations
are the outlier-heavy ones. It is also a deviation from NVIDIA's own practice — their
DeepSeek-V4 and Qwen3-Omni builds ship FP4 experts with **FP8** attention.

This notebook does three things and measures each:

1. **Remove the vision tower** — a real SigLIP tower here, unlike Gemma 4 12B.
2. **Quantize attention weights to FP8** — 230 projections, norms deliberately excluded.
3. **Pair the MTP drafter** and measure what quantization costs its acceptance rate.

Result: **32.63 GB → 23.00 GB**, and an answer to a question nobody has published.

## Setting up

In [1]:
import gc
import json
import os
import shutil
import sys
import time
from collections import Counter
from pathlib import Path

import torch
from safetensors import safe_open
from safetensors.torch import save_file

REPO = Path.home() / "work/unfuse"
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "tools"))

M = Path(os.environ.get("MODELS", Path.home() / "models"))
NVFP4 = M / "gemma-4-31b-nvfp4"      # nvidia/Gemma-4-31B-IT-NVFP4
BASE = M / "gemma-4-31b-bf16"        # google/gemma-4-31B-it
DRAFTER = M / "gemma-4-31b-drafter"  # google/gemma-4-31B-it-assistant

print(torch.__version__, torch.cuda.get_device_name(0))
p = torch.cuda.get_device_properties(0)
print(f"sm_{p.major}{p.minor}  fp8={p.major > 8 or (p.major, p.minor) >= (8, 9)}  nvfp4={p.major >= 10}")

2.13.0+cu130 NVIDIA H100 PCIe
sm_90  fp8=True  nvfp4=False


Note the last line. **NVFP4 executes only on Blackwell (sm_100+).** On this H100 the
packed weights cannot run natively — which turns out not to block verification at all,
for a reason that comes up in step 6.

## Step 1. Read what was actually shipped

Before believing a model card, read the checkpoint.

In [2]:
qc = json.loads((NVFP4 / "config.json").read_text())["quantization_config"]
print("top-level keys:", sorted(qc))
print()
for k, v in qc.items():
    if k not in ("ignore", "config_groups"):
        print(f"  {k}: {v}")
print(f"  config_groups.group_0.weights: {qc['config_groups']['group_0']['weights']}")
print(f"\n{len(qc['ignore'])} modules excluded")

top-level keys: ['config_groups', 'ignore', 'kv_cache_scheme', 'producer', 'quant_algo', 'quant_method']

  quant_algo: NVFP4
  kv_cache_scheme: {'dynamic': False, 'num_bits': 8, 'type': 'float'}
  producer: {'name': 'modelopt', 'version': '0.37.0'}
  quant_method: modelopt
  config_groups.group_0.weights: {'dynamic': False, 'num_bits': 4, 'type': 'float', 'group_size': 16}

63 modules excluded


This block is a **hybrid schema**. `quant_algo` and `quant_method` are modelopt keys;
`config_groups`, `ignore` and `kv_cache_scheme` are compressed-tensors keys. A parser that
reads only the modelopt half silently loses the group size (nested inside `config_groups`)
and the KV cache format (named differently). `unfuse`'s reader handles both.

Now count the exclusions by kind rather than reading 63 lines.

In [3]:
kind = Counter()
for pattern in qc["ignore"]:
    if "self_attn" in pattern:
        kind["attention blocks"] += 1
    elif "vision" in pattern:
        kind["vision"] += 1
    else:
        kind[pattern] += 1

for name, count in kind.most_common():
    print(f"{count:>3}  {name}")

 60  attention blocks
  2  vision
  1  lm_head


Every attention layer in the model, excluded individually.

## Step 2. Where the bytes actually are

`unfuse inspect --quant` cross-references the safetensors headers against that list, so
this is measured from the file rather than inferred from the config.

In [4]:
from unfuse import quant_report, read_inventory, read_quant_config

inv = read_inventory(NVFP4)
cfg = read_quant_config(NVFP4)
print(f"{len(inv)} tensors, {inv.params:,} parameters, {inv.nbytes / 1e9:.2f} GB")
print(f"algo {cfg.algo}  group_size {cfg.group_size}  kv cache {cfg.kv_cache_algo}\n")

print(f"{'group':<38} {'params':>16} {'GB':>7}  state")
for name, row in quant_report(inv, cfg, depth=3).items():
    print(f"{name:<38} {row['params']:>16,} {row['nbytes'] / 1e9:>7.2f}  {row['state']}")

1728 tensors, 22,169,155,796 parameters, 32.63 GB
algo NVFP4  group_size 16  kv cache FP8

group                                            params      GB  state
model.embed_vision.embedding_projection        6,193,152    0.01  BF16
model.language_model.embed_tokens         1,409,286,144    2.82  BF16
model.language_model.layers              20,184,120,740   28.66  BF16+FP8+NVFP4
model.language_model.norm                         5,376    0.00  BF16
model.vision_tower.encoder                  545,070,384    1.09  BF16
model.vision_tower.patch_embedder            24,477,696    0.05  BF16
model.vision_tower.std_bias                       1,152    0.00  BF16
model.vision_tower.std_scale                      1,152    0.00  BF16


The BF16 rows are the opportunity, and attention is by far the biggest.

One subtlety worth pausing on: `embed_tokens` reports **BF16**, not NVFP4, even though the
exclusion list never names it. It is *tied* to `lm_head`, which is excluded — so the tensor
ships under a name the pattern does not match. `unfuse` lets the dtype in the header
override the config's claim, because trusting the config here overstates how much of the
checkpoint was touched.

## Step 3. Which regime is this workload in?

Worth settling before optimising weights at all.

In [5]:
from transformers import AutoConfig

text_cfg = AutoConfig.from_pretrained(BASE).text_config
layer_types = list(text_cfg.layer_types)

# Gemma 4's layers are heterogeneous, and transformers refuses a global read of
# num_key_value_heads for exactly that reason. Assuming uniformity here overstates
# the cache by 2x, because the global layers use *fewer* KV heads than the
# sliding ones -- the opposite of the intuition.
shape = Counter()
global_bytes = sliding_bytes = 0
for kind, lc in zip(layer_types, text_cfg.per_layer_config):
    per_token = lc.num_key_value_heads * lc.head_dim * 2   # K and V, 1 byte at FP8
    shape[(kind, lc.num_key_value_heads, lc.head_dim)] += 1
    if kind == "full_attention":
        global_bytes += per_token
    else:
        sliding_bytes += per_token

for (kind, kv, hd), n in shape.most_common():
    print(f"  {n:>2} x {kind:<18} kv_heads={kv:<3} head_dim={hd}")

window = text_cfg.sliding_window
print(f"\nsliding window {window}, context {text_cfg.max_position_embeddings:,}")
print(f"global layers: {global_bytes / 1024:.1f} KB/token")
print(f"sliding layers: capped at {sliding_bytes * window / 1e9:.2f} GB regardless of length\n")
for ctx in (32_768, 131_072, 262_144):
    total = ctx * global_bytes + min(ctx, window) * sliding_bytes
    print(f"  {ctx:>7,} tokens -> {total / 1e9:>6.2f} GB")

  50 x sliding_attention  kv_heads=16  head_dim=256
  10 x full_attention     kv_heads=4   head_dim=512

sliding window 1024, context 262,144
global layers: 40.0 KB/token
sliding layers: capped at 0.42 GB regardless of length

   32,768 tokens ->   1.76 GB
  131,072 tokens ->   5.79 GB
  262,144 tokens ->  11.16 GB


The counter-intuitive part: the **sliding** layers have four times the KV heads of the
global ones, but their cache is capped by the 1024-token window, so it is a fixed ~0.4 GB
no matter how long the context runs. Only the 10 global layers grow.

At the full 262k window that comes to roughly 11 GB — about half the size of the quantized
weights, not the "as large as the weights" you would get by assuming uniform layers. Real,
but it does not dominate, and the weight savings below stay worth having at any context
length.

## Step 4. Remove the vision tower

Pure deletion, plus two renames that are easy to miss.

The tensors are named `model.language_model.*` under the multimodal class, but
`Gemma4ForCausalLM` expects `model.*`. And once that prefix moves, **every `ignore` entry
points at a module path that no longer exists** — so they match nothing, and the next tool
to read this checkpoint concludes attention was quantized when it was not.

In [6]:
VISION = ("model.vision_tower", "model.embed_vision", "model.multi_modal_projector")
OLD, NEW = "model.language_model.", "model."


def strip_and_rename(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    weight_map, total, dropped = {}, 0, 0
    for shard in sorted(src.glob("*.safetensors")):
        out = {}
        with safe_open(shard, framework="pt") as f:
            meta = f.metadata() or {}
            for name in f.keys():
                if name.startswith(VISION):
                    dropped += 1
                    continue
                new = name.replace(OLD, NEW, 1) if name.startswith(OLD) else name
                t = f.get_tensor(name)
                out[new] = t
                weight_map[new] = shard.name
                total += t.numel() * t.element_size()
        save_file(out, dst / shard.name, metadata={**meta, "format": "pt"})
        del out
        gc.collect()

    cfg = json.loads((src / "config.json").read_text())
    text = dict(cfg["text_config"])
    text["architectures"] = ["Gemma4ForCausalLM"]
    text.setdefault("model_type", "gemma4_text")
    text["dtype"] = cfg.get("dtype", "bfloat16")

    quant = cfg.get("quantization_config")
    if quant:
        quant["ignore"] = [
            p.replace(OLD, NEW, 1) if p.startswith(OLD) else p
            for p in quant.get("ignore", [])
            if not any(v in p for v in ("vision", "embed_vision"))
        ]
        text["quantization_config"] = quant
    (dst / "config.json").write_text(json.dumps(text, indent=2))

    idx = json.loads((src / "model.safetensors.index.json").read_text())
    idx["weight_map"] = weight_map
    idx["metadata"] = {"total_size": total}
    (dst / "model.safetensors.index.json").write_text(json.dumps(idx, indent=2))

    skip = {"config.json", "model.safetensors.index.json", "hf_quant_config.json",
            "preprocessor_config.json", "processor_config.json"}
    for p in src.iterdir():
        if p.is_file() and p.suffix != ".safetensors" and p.name not in skip:
            shutil.copy2(p, dst / p.name)
    return dropped, len(weight_map), total


t0 = time.time()
dropped, kept, total = strip_and_rename(NVFP4, M / "nvfp4-text")
print(f"dropped {dropped} vision tensors, kept {kept}, {total / 1e9:.2f} GB  ({time.time() - t0:.0f}s)")

dropped 356 vision tensors, kept 1372, 31.48 GB  (58s)


Do the same to the BF16 original, which becomes the reference everything is scored against.

In [7]:
t0 = time.time()
dropped, kept, total = strip_and_rename(BASE, M / "R-base-text")
print(f"reference: dropped {dropped}, kept {kept}, {total / 1e9:.2f} GB  ({time.time() - t0:.0f}s)")

reference: dropped 356, kept 832, 61.39 GB  (161s)


## Step 5. Quantize attention to FP8

Weights only. FP8 has two independent halves with very different risk: the weight half is a
pure function of the weights — no calibration data, nothing to get wrong beyond the scale
arithmetic — while the activation half needs calibration and is exactly where attention's
outliers cause trouble.

Weights-only still takes attention from 15.85 GB to 7.93 GB and still speeds up batch-1
decode, which is memory-bandwidth-bound. It gives up a prefill win, since BF16 activations
cannot feed FP8 tensor cores.

**The pattern matters more than it looks.** A bare `*self_attn*` also matches `q_norm` and
`k_norm` — an early run of this quantized all 120 of them. Those are 1-D RMSNorm scales:
no size to win, and among the most precision-sensitive weights in the model. The tool now
targets `*_proj` and additionally refuses any 1-D tensor structurally.

In [8]:
from fp8_attention import DEFAULT_PATTERNS, convert as fp8_convert

print("patterns:", DEFAULT_PATTERNS)
t0 = time.time()
stats = fp8_convert(M / "nvfp4-text", M / "SHIP", DEFAULT_PATTERNS, False, False)
print(f"\nconverted {stats['converted']} projections, skipped_1d {stats['skipped_1d']}")
print(f"  {stats['bytes_before'] / 1e9:.2f} GB -> {stats['bytes_after'] / 1e9:.2f} GB  ({time.time() - t0:.0f}s)")

patterns: ['*self_attn*_proj']


  model-00001-of-00004.safetensors: 50 converted so far


  model-00002-of-00004.safetensors: 131 converted so far


  model-00003-of-00004.safetensors: 210 converted so far



converted 230 projections, skipped_1d 0
  31.48 GB -> 23.00 GB  (78s)


  model-00004-of-00004.safetensors: 230 converted so far


230, not 240 — because 10 layers have no `v_proj` at all. Gemma 4 sets `attention_k_eq_v`,
sharing K and V in those layers. Worth checking before assuming a count is wrong.

In [9]:
counts = Counter()
for shard in sorted((M / "SHIP").glob("*.safetensors")):
    with safe_open(shard, framework="pt") as f:
        for n in f.keys():
            if f.get_slice(n).get_dtype() == "F8_E4M3" and n.endswith(".weight"):
                counts[n.split(".")[-2]] += 1
print("FP8 weights by module:", dict(counts.most_common()))

final = json.loads((M / "SHIP" / "config.json").read_text())["quantization_config"]
print("\nglobal algo      ", final["quant_algo"])
print("per-layer entries", len(final.get("quantized_layers", {})))
print("still excluded   ", [p for p in final["ignore"]][:3], "...")

FP8 weights by module: {'k_proj': 60, 'o_proj': 60, 'q_proj': 60, 'v_proj': 50}

global algo       NVFP4
per-layer entries 60
still excluded    ['lm_head'] ...


## Step 6. Verify — by expansion, not by running FP4

Two problems have the same solution.

First, **NVFP4 executes only on Blackwell**, and this is an H100. Second, and more
decisive: **transformers cannot load this checkpoint at all.** `quant_method: modelopt` is
not among the methods it supports, and it does not error on the unknown method — it ignores
the quantization config, loads the packed `uint8` tensors as dense weights, and dies on a
shape mismatch of exactly 2× the input dimension. That 2× is the tell: NVFP4 packs two
4-bit values per byte.

So expand the weights back to BF16 and score that. This is **not** an approximation for
weight-only quantization: an FP4 or FP8 kernel dequantizes the stored weight before
multiplying against BF16 activations anyway, so the expanded copy computes the same thing.
It does not simulate NVFP4's *activation* quantization — both sides come out equally
optimistic, which keeps the comparison fair for the weight change under test.

In [10]:
from dequantize import convert as expand

t0 = time.time()
print("B: NVIDIA's quantization, expanded")
print(" ", expand(M / "nvfp4-text", M / "B-nvidia-expanded", 16))
print("C: ours, expanded")
print(" ", expand(M / "SHIP", M / "C-ours-expanded", 16))
print(f"({time.time() - t0:.0f}s)")

B: NVIDIA's quantization, expanded


  model-00001-of-00004.safetensors: nvfp4=36 fp8=0


  model-00002-of-00004.safetensors: nvfp4=99 fp8=0


  model-00003-of-00004.safetensors: nvfp4=162 fp8=0


  model-00004-of-00004.safetensors: nvfp4=180 fp8=0


  {'nvfp4': 180, 'fp8': 0, 'copied': 652}
C: ours, expanded


  model-00001-of-00004.safetensors: nvfp4=36 fp8=50


  model-00002-of-00004.safetensors: nvfp4=99 fp8=131


  model-00003-of-00004.safetensors: nvfp4=162 fp8=210


  model-00004-of-00004.safetensors: nvfp4=180 fp8=230


  {'nvfp4': 180, 'fp8': 230, 'copied': 422}
(246s)


Before trusting any of it, check the expansion against the untouched original. Attention
was never quantized by NVIDIA, so after stripping and renaming it must be **bit-identical**.
The MLPs went through NVFP4 and back, so they must sit within e2m1 rounding error.

In [11]:
def wmap(path):
    return json.loads((path / "model.safetensors.index.json").read_text())["weight_map"]


def get(path, m, name):
    with safe_open(path / m[name], framework="pt") as f:
        return f.get_tensor(name).to(torch.float32)


R, B = M / "R-base-text", M / "B-nvidia-expanded"
rm, bm = wmap(R), wmap(B)
print(f"{'tensor':<46} {'max|d|':>9} {'rel':>8}")
for layer in (0, 30, 59):
    for mod in ("self_attn.q_proj", "mlp.gate_proj"):
        n = f"model.layers.{layer}.{mod}.weight"
        r, b = get(R, rm, n), get(B, bm, n)
        d = (r - b).abs().max().item()
        print(f"{n:<46} {d:>9.4f} {d / r.abs().max().item():>7.2%}")

tensor                                            max|d|      rel


model.layers.0.self_attn.q_proj.weight            0.0000   0.00%


model.layers.0.mlp.gate_proj.weight               0.0249   9.51%


model.layers.30.self_attn.q_proj.weight           0.0000   0.00%


model.layers.30.mlp.gate_proj.weight              0.0239   4.22%


model.layers.59.self_attn.q_proj.weight           0.0000   0.00%


model.layers.59.mlp.gate_proj.weight              0.0195   9.30%


Zero on attention, single-digit percent on the MLPs. Now the scale arithmetic itself — the
cheapest possible check that a dequantizer is right:

In [12]:
src = M / "nvfp4-text"
sm = wmap(src)
n = "model.layers.0.mlp.gate_proj.weight"
with safe_open(src / sm[n], framework="pt") as f:
    ws = f.get_tensor(n + "_scale").to(torch.float32)
    w2 = f.get_tensor(n + "_scale_2").to(torch.float32)

print(f"weight_scale   max = {ws.max():.4f}   (stored as float8_e4m3)")
print(f"weight_scale_2     = {w2.item():.8f}")
print(f"largest e2m1 value = 6.0")
print(f"\nproduct            = {(ws.max() * w2 * 6).item():.6f}")
print(f"original absmax    = {get(R, rm, n).abs().max().item():.6f}")

weight_scale   max = 448.0000   (stored as float8_e4m3)
weight_scale_2     = 0.00009737
largest e2m1 value = 6.0

product            = 0.261719
original absmax    = 0.261719


The two-level scale reconstructs the original range exactly. If those two numbers disagreed,
every result below would be worthless.

## Step 7. Score it — through the chat template

One more trap, and it produced a clean-looking wrong answer on the first attempt.

`gemma-4-31B-it` is **instruction-tuned**. Fed the raw prompt `"The capital of France is"`
it continues `France is France is France is` — and so does the *untouched original*, which
is what cleared the extraction of blame. Through its chat template it answers `Paris`.

Scoring in that degenerate regime inflates divergence by about an order of magnitude and
lets small perturbations move results arbitrarily, which is how a **more** quantized
checkpoint came out closer to the reference than a less quantized one. Always score an IT
model the way it is meant to be used.

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from unfuse.agreement import CHAT_PROMPTS, Agreement, _score, as_chat

tok = AutoTokenizer.from_pretrained(R)
prompts = as_chat(tok, CHAT_PROMPTS)


def logits_for(path):
    model = AutoModelForCausalLM.from_pretrained(
        path, dtype=torch.bfloat16, device_map="cuda", low_cpu_mem_usage=True).eval()
    out = {}
    with torch.no_grad():
        for p in prompts:
            out[p] = model(**tok(p, return_tensors="pt").to(model.device)).logits.float().cpu()
    ids = tok(prompts[0], return_tensors="pt").to(model.device)
    g = model.generate(**ids, max_new_tokens=6, do_sample=False)
    print(f"  {path.name} says: {tok.decode(g[0][ids['input_ids'].shape[-1]:], skip_special_tokens=True)!r}")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return out


ref = logits_for(R)
base = logits_for(B)
ours = logits_for(M / "C-ours-expanded")

[transformers] The tokenizer you are loading from '/home/ubuntu/models/R-base-text' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

  R-base-text says: 'Paris'


Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

  B-nvidia-expanded says: 'Paris'


Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

  C-ours-expanded says: 'Paris'


In [14]:
labels = [p.split("user")[-1][:34].strip() for p in prompts]
b = Agreement([_score(ref[p], base[p], lab) for p, lab in zip(prompts, labels)], label="B nvidia NVFP4")
c = Agreement([_score(ref[p], ours[p], lab) for p, lab in zip(prompts, labels)], label="C ours +FP8 attn")

print(b.report())
print()
print(c.report(b))
print(f"\nMARGINAL: {c.marginal_cost(b)}")

B nvidia NVFP4 vs reference

     top1    top5         KL     max|d|  prompt
   0.9200  0.8560  8.674e-02  1.345e+01  'What is the capital of France? An'
   0.9167  0.8000  1.318e-01  1.562e+01  'Write a Python function that retu'
   0.7826  0.8000  1.448e-01  1.062e+01  'Explain in one sentence why the s'
   0.9762  0.8381  4.403e-02  9.875e+00  'Continue the sequence and explain'
   0.9565  0.8000  4.667e-02  1.042e+01  'Translate to German: The weather'
   0.9000  0.8300  8.882e-02  9.625e+00  'State the three laws of thermodyn'

  worst top1 0.7826   worst KL 1.448e-01

C ours +FP8 attn vs reference

     top1    top5         KL     max|d|  prompt
   0.9600  0.8240  1.149e-01  1.245e+01  'What is the capital of France? An'
   0.9583  0.7917  1.716e-01  1.106e+01  'Write a Python function that retu'
   0.9130  0.8087  1.206e-01  1.066e+01  'Explain in one sentence why the s'
   1.0000  0.8143  4.649e-02  1.278e+01  'Continue the sequence and explain'
   1.0000  0.8174  5.656e-02  1.

A strict "no worse than the shipped build" gate is the right test only when a change is not
supposed to cost anything. **Ours adds quantization on top of NVIDIA's**, so it cannot pass
that gate by construction and reporting it as pass/fail would be rigged. `marginal_cost`
reports the increment instead.

Read these as indicative, not precise: six prompts is a small sample, and the top-1 column
is not cleanly ordered between the two builds — which is the sample size telling you so.

Also resist reading KL as benchmark accuracy. NVIDIA published GPQA 85.35% vs 85.80% for
this checkpoint, which *is* near-lossless. KL over a 262k vocabulary is far more sensitive
than multiple-choice accuracy, and there is no contradiction between the two.

## Step 8. Does quantizing the target cost speculative acceptance?

The open question. `google/gemma-4-31B-it-assistant` is a 0.47B drafter for this exact
target, and no NVFP4-paired build exists.

Speculative decoding is **exact by construction** — the target verifies every drafted token
and discards what it would not have produced — so this cannot change output quality. It
changes throughput, and throughput depends entirely on acceptance. The drafter was trained
against the **BF16** target; quantizing the target moves its distribution away from what the
drafter learned to predict.

Measured as **tokens per target forward pass**: the arithmetic ceiling on speedup, a
property of the two distributions rather than of any kernel.

In [15]:
from bench_spec_decode import PROMPTS, run_once

drafter = AutoModelForCausalLM.from_pretrained(
    DRAFTER, dtype=torch.bfloat16, device_map="cuda", low_cpu_mem_usage=True).eval()
print(f"drafter: {sum(p.numel() for p in drafter.parameters()) / 1e9:.2f}B")

rows = []
for label, path in [("R base BF16", R), ("B nvidia NVFP4", B), ("C ours +FP8", M / "C-ours-expanded")]:
    target = AutoModelForCausalLM.from_pretrained(
        path, dtype=torch.bfloat16, device_map="cuda", low_cpu_mem_usage=True).eval()
    alone = run_once(target, tok, PROMPTS, 96, None, label)
    spec = run_once(target, tok, PROMPTS, 96, drafter, label)
    rows.append((label, alone, spec))
    print(f"  {label:<16} {spec.tokens_per_forward:.3f} tok/fwd")
    del target
    gc.collect()
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/48 [00:00<?, ?it/s]

drafter: 0.47B


Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

  R base BF16      2.220 tok/fwd


Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

  B nvidia NVFP4   1.655 tok/fwd


Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

  C ours +FP8      1.247 tok/fwd


In [16]:
base_fwd = rows[0][2].tokens_per_forward
print(f"{'target':<18} {'tok/fwd':>9} {'vs BF16':>9}")
print("-" * 39)
for label, _, spec in rows:
    delta = 100 * (spec.tokens_per_forward - base_fwd) / base_fwd
    print(f"{label:<18} {spec.tokens_per_forward:>9.3f} {delta:>8.1f}%")

target               tok/fwd   vs BF16
---------------------------------------
R base BF16            2.220      0.0%
B nvidia NVFP4         1.655    -25.4%
C ours +FP8            1.247    -43.8%


**Monotonic in the amount of quantization**, which is what makes it credible where the
top-1 numbers were not. Acceptance is the entire speedup, so a published "up to 3×" for the
BF16 pairing does not transfer to a quantized target — NVIDIA's own build surrenders a
quarter of the benefit before anyone adds anything.

Ignore wall-clock here: transformers' assisted-generation overhead dominates at this scale,
to the point where the BF16 reference measures a speculative "speedup" below 1×. That is a
harness property, not a model property, and it is exactly why tok/fwd is the number
reported. This also uses `assistant_model` draft-model speculation rather than the native
MTP path the drafter ships for.

## Step 9. Does it actually answer questions?

In [17]:
QS = ["What is 17 * 24? Show your working briefly.",
      "Translate to German: 'The weather is cold today and I forgot my coat.'",
      "Explain in two sentences why the sky is blue."]


def answers(path):
    m = AutoModelForCausalLM.from_pretrained(
        path, dtype=torch.bfloat16, device_map="cuda", low_cpu_mem_usage=True).eval()
    out = []
    for q in QS:
        text = tok.apply_chat_template([{"role": "user", "content": q}],
                                       tokenize=False, add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to(m.device)
        with torch.no_grad():
            g = m.generate(**ids, max_new_tokens=90, do_sample=False)
        out.append(tok.decode(g[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip())
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return out


for q, r, o in zip(QS, answers(R), answers(M / "C-ours-expanded")):
    print("=" * 84)
    print("Q:", q)
    print("-" * 84)
    print("REFERENCE:", r[:300].replace("\n", " "))
    print("OURS     :", o[:300].replace("\n", " "))
    print("IDENTICAL" if r == o else "(differs)")

Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/832 [00:00<?, ?it/s]

Q: What is 17 * 24? Show your working briefly.
------------------------------------------------------------------------------------
REFERENCE: There are a few quick ways to solve this:  **Method 1: Breaking it down (Distributive Property)** 17 * 20 = 340 17 * 4 = 68 340 + 68 = **408**  **Method 2: Using 25 as a benchmark** 17 * 25 = 425 17 *
OURS     : There are a few ways to solve this. Here are two brief methods:  **Method 1: Breaking it down (Distributive Property)** 17 * 20 = 340 17 * 4 = 68 340 + 68 = **408**  **Method 2: Using 25 as a benchmark** 17 * 25 = 42
(differs)
Q: Translate to German: 'The weather is cold today and I forgot my coat.'
------------------------------------------------------------------------------------
REFERENCE: The translation is:  **"Das Wetter ist heute kalt und ich habe meinen Mantel vergessen."**
OURS     : The translation is:  **"Das Wetter ist heute kalt und ich habe meinen Mantel vergessen."**
IDENTICAL
Q: Explain in two sentences why the sky is bl

## Step 10. What we ended up with

In [18]:
def gb(path):
    return sum(p.stat().st_size for p in Path(path).glob("*.safetensors")) / 1e9


rows = [("nvidia NVFP4", gb(NVFP4)), ("+ text-only", gb(M / "nvfp4-text")),
        ("+ FP8 attention", gb(M / "SHIP"))]
start = rows[0][1]
for name, size in rows:
    print(f"{name:<20} {size:>6.2f} GB   {100 * (1 - size / start):>5.1f}% smaller")

nvidia NVFP4          32.63 GB     0.0% smaller
+ text-only           31.48 GB     3.5% smaller
+ FP8 attention       23.00 GB    29.5% smaller


## Wrapping up

**32.63 GB → 23.00 GB, 29.5% smaller**, made of NVIDIA's NVFP4 MLPs copied verbatim, 230
attention projections in FP8, and no vision tower. Generation quality is preserved.

The three things worth carrying to the next checkpoint:

- **Read the exclusion list, not the model card.** "NVFP4" here meant "the MLPs".
- **Verify by expansion.** Weight-only quantization is exactly simulatable, so accuracy
  work never needs the hardware the format targets — only throughput does.
- **Score models the way they are used.** The chat-template mistake produced a clean,
  plausible, entirely wrong answer that happened to favour our own checkpoint.

One caveat to keep in view: at 262k context the KV cache is ~11 GB, roughly half these
weights. That is worth knowing before optimising further, though it does not undo any of
the above.